# Common valid subset and cluster-wise rebalance

This notebook creates a common labeled subset that is valid for 16x16, 32x32, and 64x64 patch experiments, then balances labels within each spatial cluster.

## 1. Load paths and parameters

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.patch_dataset import (
    build_common_valid_dataframe,
    cluster_wise_balance_common_valid,
    common_valid_sample_ids,
    filter_patch_indices_to_selected_ids,
    load_patch_indices,
    make_common_valid_balanced_sample_id_table,
    save_common_valid_balanced_outputs,
)

patch_dir = PROJECT_ROOT / "data/processed/patches"
patch_index_paths = {
    16: patch_dir / "labeled_patch_index_ps16.csv",
    32: patch_dir / "labeled_patch_index_ps32.csv",
    64: patch_dir / "labeled_patch_index_ps64.csv",
}
selected_table_path = patch_dir / "common_valid_balanced_sample_ids.csv"
balanced_output_paths = {
    16: patch_dir / "labeled_patch_index_ps16_common_balanced.csv",
    32: patch_dir / "labeled_patch_index_ps32_common_balanced.csv",
    64: patch_dir / "labeled_patch_index_ps64_common_balanced.csv",
}
random_seed = 42

print("Input patch index paths:")
for patch_size, path in patch_index_paths.items():
    print(f"{patch_size}: {path}")
print(f"Selected ID table: {selected_table_path}")

## 2. Load patch indices and inspect original valid counts

In [ ]:
patch_indices = load_patch_indices(patch_index_paths)
for patch_size, patch_index in patch_indices.items():
    valid = patch_index.loc[patch_index["valid_patch"].astype(bool)]
    print(f"Patch size {patch_size}")
    print(f"original valid count: {len(valid)}")
    print("original valid counts by label:")
    print(valid["label"].value_counts().sort_index().to_string())
    print("")

## 3. Compute common valid sample IDs

In [ ]:
common_ids = common_valid_sample_ids(patch_indices)
common_valid_df = build_common_valid_dataframe(patch_indices, common_ids, reference_patch_size=16)

print(f"Common valid sample count before rebalancing: {len(common_valid_df)}")
print("Common valid counts by cluster_id and label before rebalancing:")
print(pd.crosstab(common_valid_df["cluster_id"], common_valid_df["label"]).to_string())

## 4. Cluster-wise label balancing

In [ ]:
selected_common_valid = cluster_wise_balance_common_valid(
    common_valid_df,
    random_seed=random_seed,
)
selected_table = make_common_valid_balanced_sample_id_table(selected_common_valid)
selected_ids = set(selected_table["sample_id"].astype(str))

print("Final selected counts by cluster_id and label after rebalancing:")
print(pd.crosstab(selected_common_valid["cluster_id"], selected_common_valid["label"]).to_string())
print(f"Final total selected samples: {len(selected_table)}")

## 5. Filter all patch sizes to identical selected IDs

In [ ]:
filtered_patch_indices = filter_patch_indices_to_selected_ids(patch_indices, selected_ids)
for patch_size, patch_index in filtered_patch_indices.items():
    print(f"Patch size {patch_size}: {len(patch_index)} selected rows")
    print(pd.crosstab(patch_index["cluster_id"], patch_index["label"]).to_string())
    print("")

## 6. Save common balanced outputs

In [ ]:
saved_paths = save_common_valid_balanced_outputs(
    selected_table,
    filtered_patch_indices,
    selected_table_path,
    balanced_output_paths,
)

print("Output file paths:")
for key, path in saved_paths.items():
    print(f"{key}: {path}")